In [5]:
import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import medfilt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [6]:
df = pd.read_excel("/Volumes/Samsung_T5/first_iterations/final_combined_dataset_only_cough.xlsx", header=1)

# Normalise column names (strip whitespace just in case)
df.columns = df.columns.str.strip()

print(f"Dataset shape : {df.shape}")
print(f"Columns       : {df.columns.tolist()}")
print()
print("Disease distribution:")
print(df['disease'].value_counts())

Dataset shape : (2780, 9)
Columns       : ['id', 'age', 'gender', 'smoker', 'cold_present', 'cough_present', 'fever_present', 'disease', 'cough_path']

Disease distribution:
disease
healthy    1715
covid       545
asthma      354
copd        166
Name: count, dtype: int64


In [7]:
INPUT_CSV       =  df 
OUTPUT_DIR = "/Volumes/Samsung_T5/first_iterations/result "
OUTPUT_CSV = "/Volumes/Samsung_T5/respiratory_classification_project/dataset_segmented.csv"
# Audio settings
SR              = 22050     # sample rate — standard for audio ML
FRAME_LEN       = 512       # ~23ms window for energy calculation
HOP_LEN         = 128       # ~6ms step between frames
 
# VAD thresholds (from energy envelope plots)
KAGGLE_THRESH   = 0.02      # works for kaggle — clean recordings
COSWARA_THRESH  = 0.05      # slightly higher for coswara — more background noise
 
# Cough segment duration limits
COUGH_MIN_SEC   = 0.3       # anything shorter is noise/click, skip it
COUGH_MAX_SEC   = 3.0       # anything longer is not a single cough, skip it
 
# Vowel segment duration limits
VOWEL_MIN_SEC   = 1.0       # vowel must be at least 1 second
VOWEL_MAX_SEC   = 6.0       # vowel rarely exceeds 6 seconds
 
# Padding around each detected segment
PAD_SEC         = 0.05      # 50ms padding to capture attack/release

In [8]:
import librosa 
def detect_events(y, sr, thresh):
    """
    Given an audio signal y:
    1. Compute RMS energy per frame
    2. Normalize 0-1
    3. Apply threshold to get binary VAD mask
    4. Find contiguous active regions (events)
    Returns list of (start_frame, end_frame) tuples
    """
    # Compute RMS energy
    rms = librosa.feature.rms(y=y, frame_length=FRAME_LEN, hop_length=HOP_LEN)[0]
 
    # Normalize between 0 and 1
    rms_norm = rms / (rms.max() + 1e-9)
 
    # Binary mask: True = sound, False = silence
    active = rms_norm > thresh
 
    # Find contiguous active regions
    events = []
    in_event = False
    start = 0
    for i, a in enumerate(active):
        if a and not in_event:        # silence → sound: mark start
            start = i
            in_event = True
        elif not a and in_event:      # sound → silence: mark end
            events.append((start, i))
            in_event = False
    if in_event:                      # handle sound right at end of file
        events.append((start, len(active) - 1))
 
    return events

In [9]:
def segment_audio(audio_path, patient_id, disease, audio_type, output_dir):
    """
    Loads one audio file, runs VAD, saves valid segments.
 
    audio_type: "cough" or "vowel"
    Returns list of dicts (one per saved segment)
    """
    segments = []
 
    # Detect which dataset this file is from
    is_coswara = "coswara-data" in audio_path
    thresh     = COSWARA_THRESH if is_coswara else KAGGLE_THRESH
    source     = "coswara" if is_coswara else "kaggle"
 
    # Set duration limits based on audio type
    if audio_type == "cough":
        min_sec = COUGH_MIN_SEC
        max_sec = COUGH_MAX_SEC
    else:  # vowel — one long sustained sound, wider range
        min_sec = VOWEL_MIN_SEC
        max_sec = VOWEL_MAX_SEC
 
    # Load audio
    try:
        y, sr = librosa.load(audio_path, sr=SR, mono=True)
    except Exception as e:
        print(f"  ❌ Could not load {audio_path}: {e}")
        return segments
 
    # Detect events using VAD
    events    = detect_events(y, sr, thresh)
    pad_frames = int(PAD_SEC * SR / HOP_LEN)
 
    # Loop through each detected event
    for idx, (s, e) in enumerate(events):
 
        # Add padding around event
        s_pad = max(0, s - pad_frames)
        e_pad = min(len(librosa.feature.rms(
            y=y, frame_length=FRAME_LEN, hop_length=HOP_LEN)[0]) - 1,
            e + pad_frames)
 
        # Convert frames to samples
        start_sample = s_pad * HOP_LEN
        end_sample   = e_pad * HOP_LEN
        duration     = (end_sample - start_sample) / SR
 
        # Skip segments outside duration range
        if duration < min_sec or duration > max_sec:
            continue
 
        # Extract segment
        segment  = y[start_sample:end_sample]
 
        # Save segment as wav file
        filename = f"{patient_id}_{audio_type}_{idx}.wav"
        out_path = os.path.join(output_dir, filename)
        sf.write(out_path, segment, SR)
 
        segments.append({
            "patient_id"   : patient_id,
            "disease"      : disease,
            "audio_type"   : audio_type,   # "cough" or "vowel"
            "segment_path" : out_path,
            "duration_sec" : round(duration, 3),
            "segment_idx"  : idx,
            "source"       : source        # "kaggle" or "coswara"
        })
 
    return segments

In [10]:
def segment_audio(audio_path, patient_id, disease, audio_type, output_dir):
    """
    Loads one audio file, runs VAD, saves valid segments.
 
    audio_type: "cough" or "vowel"
    Returns list of dicts (one per saved segment)
    """
    segments = []
 
    # Detect which dataset this file is from
    is_coswara = "coswara-data" in audio_path
    thresh     = COSWARA_THRESH if is_coswara else KAGGLE_THRESH
    source     = "coswara" if is_coswara else "kaggle"
 
    # Set duration limits based on audio type
    if audio_type == "cough":
        min_sec = COUGH_MIN_SEC
        max_sec = COUGH_MAX_SEC
    else:  # vowel — one long sustained sound, wider range
        min_sec = VOWEL_MIN_SEC
        max_sec = VOWEL_MAX_SEC
 
    # Load audio
    try:
        y, sr = librosa.load(audio_path, sr=SR, mono=True)
    except Exception as e:
        print(f"  ❌ Could not load {audio_path}: {e}")
        return segments
 
    # Detect events using VAD
    events    = detect_events(y, sr, thresh)
    pad_frames = int(PAD_SEC * SR / HOP_LEN)
 
    # Loop through each detected event
    for idx, (s, e) in enumerate(events):
 
        # Add padding around event
        s_pad = max(0, s - pad_frames)
        e_pad = min(len(librosa.feature.rms(
            y=y, frame_length=FRAME_LEN, hop_length=HOP_LEN)[0]) - 1,
            e + pad_frames)
 
        # Convert frames to samples
        start_sample = s_pad * HOP_LEN
        end_sample   = e_pad * HOP_LEN
        duration     = (end_sample - start_sample) / SR
 
        # Skip segments outside duration range
        if duration < min_sec or duration > max_sec:
            continue
 
        # Extract segment
        segment  = y[start_sample:end_sample]
 
        # Save segment as wav file
        filename = f"{patient_id}_{audio_type}_{idx}.wav"
        out_path = os.path.join(output_dir, filename)
        sf.write(out_path, segment, SR)
 
        segments.append({
            "patient_id"   : patient_id,
            "disease"      : disease,
            "audio_type"   : audio_type,   # "cough" or "vowel"
            "segment_path" : out_path,
            "duration_sec" : round(duration, 3),
            "segment_idx"  : idx,
            "source"       : source        # "kaggle" or "coswara"
        })
 
    return segments

In [11]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
all_segments = []
failed       = 0
 
print(f"Processing {len(df)} recordings...\n")
 
for i, row in df.iterrows():
    pid     = row["id"]
    disease = row["disease"]
 
    cough_path = row["cough_path"]
 
    # Skip rows with missing disease label
    if pd.isna(disease):
        failed += 1
        continue
 
    # --- Process cough ---
    if pd.notna(cough_path) and os.path.exists(str(cough_path)):
        cough_segs = segment_audio(cough_path, pid, disease, "cough", OUTPUT_DIR)
        all_segments.extend(cough_segs)
    else:
        failed += 1
 
    # Progress update every 100 patients
    if (i + 1) % 100 == 0:
        print(f"  [{i+1}/{len(df)}] Segments so far: {len(all_segments)}")


Processing 2780 recordings...

  [100/2780] Segments so far: 1400
  [200/2780] Segments so far: 2635
  [300/2780] Segments so far: 3832
  [400/2780] Segments so far: 5014
  [500/2780] Segments so far: 6044
  [600/2780] Segments so far: 6715
  [700/2780] Segments so far: 6991
  [800/2780] Segments so far: 7221
  [900/2780] Segments so far: 7514
  [1000/2780] Segments so far: 7780
  [1100/2780] Segments so far: 8123
  [1200/2780] Segments so far: 8522
  [1300/2780] Segments so far: 8918
  [1400/2780] Segments so far: 9227
  [1500/2780] Segments so far: 9524
  [1600/2780] Segments so far: 9883
  [1700/2780] Segments so far: 10199
  [1800/2780] Segments so far: 10548
  [1900/2780] Segments so far: 10858
  [2000/2780] Segments so far: 11186
  [2100/2780] Segments so far: 11519
  [2200/2780] Segments so far: 11844
  [2300/2780] Segments so far: 12211
  [2400/2780] Segments so far: 12529
  [2500/2780] Segments so far: 12851
  [2600/2780] Segments so far: 13170
  [2700/2780] Segments so far: 1

In [12]:
df_seg = pd.DataFrame(all_segments)
df_seg.to_csv(OUTPUT_CSV, index=False)
 
print(f"\n✅ Done!")
print(f"   Total patients       : {len(df)}")
print(f"   Failed / skipped     : {failed}")
print(f"   Total segments saved : {len(df_seg)}")
print(f"   Avg per patient      : {len(df_seg) / max(len(df)-failed, 1):.1f}")
 
print(f"\nDisease distribution:")
print(df_seg["disease"].value_counts())
 
print(f"\nAudio type breakdown:")
print(df_seg["audio_type"].value_counts())
 
print(f"\nSource breakdown:")
print(df_seg["source"].value_counts())
 
print(f"\nSample output:")
print(df_seg.head(3).to_string())


✅ Done!
   Total patients       : 2780
   Failed / skipped     : 0
   Total segments saved : 13682
   Avg per patient      : 4.9

Disease distribution:
disease
healthy    6744
asthma     3339
copd       2044
covid      1555
Name: count, dtype: int64

Audio type breakdown:
audio_type
cough    13682
Name: count, dtype: int64

Source breakdown:
source
coswara    7125
kaggle     6557
Name: count, dtype: int64

Sample output:
      patient_id disease audio_type                                                            segment_path  duration_sec  segment_idx  source
0  b87ea0dd760fa  asthma      cough  /Volumes/Samsung_T5/first_iterations/result /b87ea0dd760fa_cough_0.wav         0.302            0  kaggle
1  b87ea0dd760fa  asthma      cough  /Volumes/Samsung_T5/first_iterations/result /b87ea0dd760fa_cough_1.wav         0.673            1  kaggle
2  b87ea0dd760fa  asthma      cough  /Volumes/Samsung_T5/first_iterations/result /b87ea0dd760fa_cough_2.wav         0.424            2  kaggle
